# 02 - M1: TF-IDF + One-vs-Rest Logistic Regression
Word 1-2-grams (VnCoreNLP-segmented) + char_wb 1-5-grams, sublinear TF; C and class_weight chosen on
validation Macro-F1 @ tau=0.5. Config: `configs/tfidf_lr.yaml`.

In [1]:
# ---- Setup: works both locally and on Google Colab --------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # EDIT: folder on your Drive that contains src/, configs/, resources/ and data/{train,val,test}.csv
    PROJECT_DIR = "/content/drive/MyDrive/vigoemotions_midterm"
    os.chdir(PROJECT_DIR)
    subprocess.run("pip -q install -r requirements-colab.txt", shell=True, check=True)
    # Java is needed by VnCoreNLP (PhoBERT word segmentation); falls back to pyvi otherwise
    if subprocess.run("java -version", shell=True, capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y default-jre-headless > /dev/null", shell=True)
else:
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
os.environ["PYTHONPATH"] = os.path.abspath("src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("project dir:", os.getcwd())

import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
T = lambda name: pd.read_csv(f"results/tables/{name}")          # noqa: E731
F = lambda name, w=900: display(Image(f"results/figures/{name}", width=w))  # noqa: E731

project dir: /Users/tam/Documents/projects/ATI/midterm


In [2]:
RETRAIN = not os.path.exists("results/runs/tfidf_lr/summary.json")
if RETRAIN:
    !python -m vigo train-tfidf --config configs/tfidf_lr.yaml

In [3]:
import json
cfg = json.load(open("results/runs/tfidf_lr/config.json"))
display(pd.DataFrame(cfg["search_results"]))
print("selected:", cfg["selected"])
summ = json.load(open("results/runs/tfidf_lr/summary.json"))
pd.Series({k: v for k, v in summ.items() if k.startswith("test_") or k in ("total_params", "train_time_s")})

,C,class_weight,val_macro_f1@0.5
0,1,None,0.361453
1,1,balanced,0.517372
2,2,None,0.410340
3,2,balanced,0.527176
4,4,None,0.444525
5,4,balanced,0.531338
6,8,None,0.463389
7,8,balanced,0.530796
8,16,None,0.474390
9,16,balanced,0.527486


selected: {'C': 4, 'class_weight': 'balanced'}


total_params                  2.136316e+06
train_time_s                  6.130623e+00
test_macro_f1                 5.419180e-01
test_micro_f1                 5.576106e-01
test_weighted_f1              5.605683e-01
test_samples_f1               5.358522e-01
test_hamming_loss             6.653881e-02
test_lrap                     6.900487e-01
test_map_macro                5.603441e-01
test_map_micro                5.958517e-01
test_subset_accuracy          1.649734e-01
test_macro_precision          4.969023e-01
test_macro_recall             6.003467e-01
test_micro_precision          5.095528e-01
test_micro_recall             6.156773e-01
test_macro_f1_27emo           5.543591e-01
test_avg_pred_cardinality     2.304306e+00
test_empty_prediction_rate    4.402516e-02
dtype: float64